# 10 – Explorer

Interactive view of the dataset built by `01_build_dataset.ipynb`. Run all cells, then use the widgets.

* **Company** – trends of selected metrics for one company
* **Compare** – one metric across several companies
* **Snapshot** – latest fiscal year overview, sortable

In [1]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

from stockpred.config import load_settings
from stockpred.plots import LABELS, trend_figure, compare_figure, snapshot_table

settings = load_settings()
panel = pd.read_parquet(settings.processed_dir / "fundamentals_annual.parquet")
universe = pd.read_csv(settings.processed_dir / "universe.csv")
print(f"{panel['ticker'].nunique()} companies, FY{panel['fiscal_year'].min()}–{panel['fiscal_year'].max()}")

49 companies, FY2010–2026


In [2]:
# --- widgets -----------------------------------------------------------------
ticker_options = [(f"{r.ticker} – {r.name}", r.ticker) for r in universe.itertuples()]
metric_options = [(label, key) for key, label in LABELS.items() if key in panel.columns]
default_metrics = ["revenue", "net_income", "free_cash_flow", "eps_diluted",
                   "gross_margin", "operating_margin", "roe", "revenue_growth"]

ticker_dd = widgets.Dropdown(options=ticker_options, description="Company", layout={"width": "420px"})
metrics_sel = widgets.SelectMultiple(options=metric_options, value=default_metrics,
                                     description="Metrics", rows=10, layout={"width": "420px"})
company_out = widgets.Output()

compare_sel = widgets.SelectMultiple(options=ticker_options, value=[t for _, t in ticker_options[:5]],
                                     description="Companies", rows=10, layout={"width": "420px"})
compare_metric = widgets.Dropdown(options=metric_options, value="operating_margin", description="Metric")
compare_out = widgets.Output()

snapshot_out = widgets.Output()


# --- callbacks: re-draw the chart whenever a widget value changes ---------------
def draw_company(*_):
    with company_out:
        company_out.clear_output(wait=True)
        trend_figure(panel, ticker_dd.value, list(metrics_sel.value)).show()


def draw_compare(*_):
    with compare_out:
        compare_out.clear_output(wait=True)
        compare_figure(panel, list(compare_sel.value), compare_metric.value).show()


for w in (ticker_dd, metrics_sel):
    w.observe(draw_company, names="value")
for w in (compare_sel, compare_metric):
    w.observe(draw_compare, names="value")

with snapshot_out:
    snap = snapshot_table(panel, universe)
    display(snap.style.format({
        "revenue_usd_bn": "{:,.1f}", "market_cap_usd_bn": "{:,.0f}",
        "revenue_cagr_5y": "{:.1%}", "gross_margin": "{:.1%}", "operating_margin": "{:.1%}",
        "net_margin": "{:.1%}", "fcf_margin": "{:.1%}", "roe": "{:.1%}",
    }, na_rep="–"))

tabs = widgets.Tab(children=[
    widgets.VBox([widgets.HBox([ticker_dd, metrics_sel]), company_out]),
    widgets.VBox([widgets.HBox([compare_sel, compare_metric]), compare_out]),
    snapshot_out,
])
for i, title in enumerate(["Company", "Compare", "Snapshot"]):
    tabs.set_title(i, title)

draw_company()
draw_compare()
display(tabs)

### How to read the charts
* Values are **as first reported** in each 10-K (later restatements ignored) – this is what investors knew at the time.
* EPS is split-adjusted to today's share basis.
* ROE can exceed 100 % for companies with tiny equity due to buybacks (e.g. Apple) – not an error.
* Missing bars = the company did not report that XBRL concept in that year (see data quality check in notebook 01).